## **Learning Objectives**
### In this notebook, we will build a team of AI agents that work together to achieve a goal.

## **Topic**
#    **|to**
## **Researcher AI Agent receives a topic (Research Findings using tavily_search tool)**
#     **|to**
## **Analyst AI Agent (carries out analysis and key insights)**
#      **|to**
## **Writer AI Agent (writes reports)**
#       **|to**
## **Manager Function**


### 1. Create modular AI agents (Researcher, Analyst, Writer) with specialized roles for information gathering, analysis, and reporting.
### 2. Implement agent collaboration by passing outputs from one agent to another in a structured workflow.
### 3. Customize AI agent behaviour by rewriting instructions to adapt outputs to specific languages (Multiple Languages) or formats (e.g, French report generation)
### 4. Build a manager function to orchestrate multi-agent workflows from input to final deliverable.
### 5. Design creative multi-agent pipelines for advertising campaigns involving role-specific agents like creative Director, Strategist and Copywriter.

In [ ]:
!pip install -q openai-agents==0.2.2 python-dotenv requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade pydantic

In [ ]:
!pip install python-dotenv

In [ ]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from agents import Agent, Runner, function_tool, SQLiteSession
from typing_extensions import TypedDict
from pydantic import BaseModel

from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

tavily_api_key = userdata.get("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = tavily_api_key

print(f"Retrieved API Key: {openai_api_key[:5]}...{openai_api_key[-5:]}" if openai_api_key else "OpenAI API Key not found or is None")
openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI Client successfully configured.")

# The Tavily client initialization and associated print statements are now handled
# in a separate cell (cell 86136d1b) to keep concerns separated and avoid NameErrors.

Retrieved API Key: sk-pr...c_bMA
OpenAI Client successfully configured.


In [ ]:
!pip install -q tavily-python

In [ ]:
from tavily import TavilyClient

# Configure the Tavily client separately
tavily_client = TavilyClient(api_key=tavily_api_key)
print("Tavily Client successfully configured.")

Tavily Client successfully configured.


In [ ]:
def print_markdown(text):
    display(Markdown(text))

In [ ]:
import json # import the json module for handling json data
from typing_extensions import TypedDict # import TypedDict for type hinting
from agents import function_tool

# A typedDict describes the expected keys and value types for a dict
# Here: a search "query" (string) and "max_results" (int)

class TavilySearchParams(TypedDict):
    query: str
    max_results: int

# This decorator (from OpenAI Agents SDK) registers the function as a tool that an agent can call
@function_tool
def tavily_search(params: TavilySearchParams) -> str:

  # The web address (endpoint) for sending the search request to Tavily (Tavily API endpoint)
  url = "https://api.tavily.com/search"

  # Tell the API that we are sending json data
  headers = {"Content-Type": "application/json"}

  # Build the request body:
  # api_key: Your Tavily API key (assumed defined elsewhere as tavily_api_key)
  # query: taken from the params dict
  # max_results: use provided value or default to 3 if missing
  payload = {
      "api_key": tavily_api_key, # Use the Tavily API key from environment
      "query": params["query"], # The search query from params
      "max_results": params.get("max_results", 3), # Use max_results from params, default to 3 if not provided.
  }

  # Send the POST request with JSON body and headers
  response = requests.post(url, json = payload, headers = headers)

  # Check if the search worked (200 is ok)
  if response.status_code == 200:  # If the request was successful
      results = response.json().get ("results", []) # Extract the results'list from the response json

      # Build a summary string with each result's title and content, numbered
      summary = "\n".join([f".{r['title']}: {r['content']}" for r in results]) # Return summary or fallback message
      return summary if summary else "No relevant results found"
  else:
      return f"Tavily API error: {response.status_code}" # Return error message with status code if request failed

print("Tavily search tool ready.")

Tavily search tool ready.


## **Define two AI Agents (Researcher and Analyst)**

### **Let's define Researcher AI Agent**

In [ ]:
# Let's define our Researcher AI Agent

# Define a Pydantic model for the agent's output.
# This ensures the agent's final output will have a single field: "summary" (string).
# Note that this will be used in both the researcher and analyst agents.
class AnalysisSummary(BaseModel):
    summary: str

# Create an AI Agent called "Researcher"
researcher_agent = Agent(name = "Researcher",
                        instructions = """

## Context
You are a research agent with access to the Tavily search tool.

## Instruction
Given a user's query, use the Tavily search tool to find relevant information and summarize the key findings.

## Input
- A research query from the user

## Output
-A summary of key findings in a maximum of 5 bullet points.
""",
    model = "gpt-4.1-mini",
    tools = [tavily_search],
    output_type = AnalysisSummary)
print("Researcher AI Agent is now ready")

Researcher AI Agent is now ready


### **Let's run the Researcher AI Agent**

In [ ]:
researcher_result = await Runner.run(researcher_agent, "Why is Labubu so popular and what are the rarest collectibles?")
print_markdown(f"### Agent's Answer\n{researcher_result.final_output.summary}")

### Agent's Answer
- Labubu's popularity surged globally, especially after endorsements by celebrities like Lisa from BLACKPINK, and the character's appeal as a cute, nostalgic collectible.
- The brand Pop Mart drove the popularity with unique blind box releases and collaborations, generating significant revenue (e.g., $677 million in the first half of 2025).
- Labubu appeals mostly to young adults, especially females aged 25-34, with a strong presence in East and Southeast Asia.
- Rare Labubu collectibles include "Secret" figures with very low pull rates (1 in 72 or 1 in 144) such as the Black Rainbow Labubu, the "Pink Microwave" Labubu, and exclusive artist prototype pieces, some worth thousands to over a million yuan.
- Authenticity is crucial due to widespread counterfeits, identifiable by QR codes and branding; rare figures are highly sought after as both art and investment.

### **Let's define Analyst AI Agent**

In [ ]:
analyst_agent = Agent(name = "Analyst",
                      instructions = """

## Context
You are an analyst who receives research notes generated by the reseach agent.

## Instruction
Given th research notes, analyze the content and extract key trends, risks or insights.

## Input
Research notes (summaries of findings from the agent

## Output
-A concise analysis (no more than 2 paragraphs) highlighting key trends, risks, or insights
""",
    model = "gpt-4.1-mini",
    output_type = AnalysisSummary,
)
print("Analyst AI Agent is now ready")


Analyst AI Agent is now ready


## **Practice Opportunity**
### Use the researcher_agent and analyst_agent to analyze a specific research question or topic.
### -Provide a research question (e.g, "What are the latest trends in Batteries for Electic Vehicles?")
### -Call the researcher_agent with research questions to obtain key findings.
### -Pass the researcher's summary to the analyst_agent to extract key trends, risks, or insights.
### -Show how you would structure the conversation and pass information between the agents.

In [ ]:
researcher_result = await Runner.run(researcher_agent, "What are the latest trends in Batteries for Electic Vehicles?")

In [ ]:
analyst_result = await Runner.run(analyst_agent, researcher_result.final_output.summary)

print_markdown(f"### Agent's Answer\n{analyst_result.final_output.summary}")

### Agent's Answer
The electric vehicle (EV) battery landscape is rapidly evolving with the adoption of ultra-fast charging and high-voltage (800V) battery architectures that enhance charging speed and efficiency while reducing battery weight. Global battery manufacturing capacity is expanding, particularly in China, the US, and Europe, to fulfill the surging demand driven by increasing EV sales. Emerging technologies such as sodium-ion batteries offer advantages in cost and extreme temperature performance but currently have lower energy density than lithium-ion counterparts. Meanwhile, solid-state batteries promise a transformative impact by providing safer, smaller, and more energy-dense solutions through the use of solid electrolytes. Additionally, trends like the adoption of lithium iron phosphate (LFP) batteries for affordability and sustainability, manganese-rich cathodes to lessen reliance on cobalt while boosting energy density, and improvements in battery recycling and circular supply chains highlight the focus on cost reduction, performance enhancement, and environmental sustainability in the battery industry.

### **Let's define Writer AI Agent (Executive Report Generator)**

In [ ]:
# Define a pydantic model for the agent's output.
# This specifies exactly what fields the writer agent should return.

class FinalReport(BaseModel):
    short_summary: str  # A brief 3-3 sentence executive summary.
    markdown_report: str  # A detailed report in markdown format (at least 500 words)
    follow_up_questions: list[str]  # Alist of 3-5 suggested follow-up research questions.

# Create an AI agent called "Writer"
writer_agent = Agent(name = "Writer",
                     instructions = """
You are a senior market analyst tasked with generating an executive-level research report.

## Context
You will receive:
-The original user query (the research question or topic of interest)
-Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent'
 (for data analysis and key insights)

 You may call these helper tools as needed to gather additional information or clarify findings.

 ## Instructions
 Your job is to synthesize all available information and produce the following outputs:
 1.**Executive Summary:**Write a concise summary (2-3 sentences) highlighting the most important findings relevant to
  the original user query and the data gathered from the search snippets and analyst summaries.
  This summary should focus on capturing the essence of the overall research, giving readers a
  quick understanding of the report's key insights.
2.**Detailed Markdown Report:**Compose a comprehensive structured report in markdown format (at least 500 words)
  covering the analysis of the provided search snippets and summaries. Organize the content with clear headings
  and logical flows. Include sections such as:

Introduction: Briefly introduce the topic and the purpose of the report.
Methodology: Explain how the information was gathered and processed.
Findings: Present the key findings from the analysis, detailing patterns or significant insights from the data.
Discussion: Analyze what these findings mean in the context of the original query.
Conclusion: Summarize the overall insights and any recommendations based on the findings.

## Input
-The original user query.
-Summaries and analyses from the helper tools.

## Output
Return a structured object with:
-short_summary: The executive summary.
-markdown_report: The detailed markdown report.
-follow_up_questions: A list of 3-5 follow_up research questions.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport,

)

print("Writer AI Agent is now ready")

Writer AI Agent is now ready


## **Practice Opportunity**
### -Rewrite the writer agent's instructions so that it generates all outputs entirely in French (Executive Summary, Detailed Markdown Report, and follow-up Research Questions.

In [ ]:
# Define a pydantic model for the agent's output.
# This specifies exactly what fields the writer agent should return.

class FinalReport(BaseModel):
    short_summary: str  # A brief 3-3 sentence executive summary.
    markdown_report: str  # A detailed report in markdown format (at least 500 words)
    follow_up_questions: list[str]  # Alist of 3-5 suggested follow-up research questions.

# Create an AI agent called "Writer"
writer_agent = Agent(name = "Writer",
                     instructions = """
You are a senior market analyst tasked with generating an executive-level research report **in French**.

## Context
You will receive:
-The original user query (the research question or topic of interest)
-Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent'
 (for data analysis and key insights)

 You may call these helper tools as needed to gather additional information or clarify findings.

 ## Instructions
 Your job is to synthesize all available information and produce the following outputs in **French**:
 1.**Executive Summary:**Write a concise summary (2-3 sentences) in French highlighting the most important findings relevant to
  the original user query and the data gathered from the search snippets and analyst summaries.
  This summary should focus on capturing the essence of the overall research, giving readers a
  quick understanding of the report's key insights.

2.**Detailed Markdown Report:**Compose a comprehensive, well-structured report in French in markdown format (at least 500 words)
  covering the analysis of the provided search snippets and summaries.

3.**Follow-up Research Questions: **Suggest 3-5 thoughtful follow-up questions in French for further investigation.

Organize the content with clear headings and logical flows in French. Include sections such as:
Introduction: Briefly introduce the topic and the purpose of the report.
Methodology: Explain how the information was gathered and processed.
Findings: Present the key findings from the analysis, detailing patterns or significant insights from the data.
Discussion: Analyze what these findings mean in the context of the original query.
Conclusion: Summarize the overall insights and any recommendations based on the findings.

## Input
-The original user query.
-Summaries and analyses from the helper tools.

## Output
Return a structured object with:
-short_summary: The executive summary in French.
-markdown_report: The detailed markdown report in French.
-follow_up_questions: A list of 3-5 follow_up research questions in French.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport,

)

print("Writer AI Agent in French is now ready")

Writer AI Agent in French is now ready


### **Build a Manager Function to Orchestrate the Full Pipeline**

In [ ]:
# Create a session to store and share information between agents.
session = SQLiteSession("research_agent_practice")

# Define an async function that takes the user's question and run the agents.
async def manager_run(user_query: str):
    # Show the user's question in a nice markdown format.
    print_markdown(f"**User's Request:** {user_query}")

    # Step 1: Run the researcher agent to gather information about the user's query.
    researcher_result = await Runner.run(researcher_agent, user_query, session = session)
    research_summary = researcher_result.final_output.summary

    # Step 2: Run the analyst agent to analyze the research findings.
    analyst_result = await Runner.run(analyst_agent, research_summary, session = session)
    analyst_summary = analyst_result.final_output.summary  # Get the short analysis summary.

    # Step 3: Prepare the combined input for the writer agent.
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summaru: {research_summary}\n"
        f"Analysis Summary: {analyst_summary}"
    )

    # Step 4: Run the writer agent to create the final report.
    writer_result = await Runner.run(writer_agent, input_for_writer, session = session)
    final_output: FinalReport = writer_result.final_output  # The final report from the writer

    # Step 5: Display the results in a nice, clear format.
    print_markdown("---")
    print_markdown(f"### **short summary:**\n{final_output.short_summary}")  # Quick Overview.
    print_markdown("\n\n------------\n\n")
    print_markdown(f"### **Full Report (Markdown):**\n{final_output.markdown_report}")  # Detailed Report.
    print_markdown("\n\n------------\n\n")
    print_markdown("### **Follow-up-Questions:**\n-" + "\n-".join(final_output.follow-up-questions)
    )
    # Any extra question the user might explore.
    print_markdown("\n\n-------------\n\n")

### **Let's run the Manager Function**

In [ ]:
import sys

# Redefining the manager_run function with the corrected attribute name (using underscores instead of hyphens)
async def manager_run(user_query: str):
    # Show the user's question in a nice markdown format.
    print_markdown(f"**User's Request:** {user_query}")

    # Step 1: Run the researcher agent to gather information about the user's query.
    researcher_result = await Runner.run(researcher_agent, user_query, session = session)
    research_summary = researcher_result.final_output.summary

    # Step 2: Run the analyst agent to analyze the research findings.
    analyst_result = await Runner.run(analyst_agent, research_summary, session = session)
    analyst_summary = analyst_result.final_output.summary  # Get the short analysis summary.

    # Step 3: Prepare the combined input for the writer agent.
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summary: {research_summary}\n"
        f"Analysis Summary: {analyst_summary}"
    )

    # Step 4: Run the writer agent to create the final report.
    writer_result = await Runner.run(writer_agent, input_for_writer, session = session)
    final_output: FinalReport = writer_result.final_output  # The final report from the writer

    # Step 5: Display the results in a nice, clear format.
    print_markdown("---")
    print_markdown(f"### **short summary:**\n{final_output.short_summary}")  # Quick Overview.
    print_markdown("\n\n------------\n\n")
    print_markdown(f"### **Full Report (Markdown):**\n{final_output.markdown_report}")  # Detailed Report.
    print_markdown("\n\n------------\n\n")
    # FIXED: Changed final_output.follow-up-questions to final_output.follow_up_questions
    print_markdown("### **Follow-up-Questions:**\n-" + "\n-".join(final_output.follow_up_questions))
    # Any extra question the user might explore.
    print_markdown("\n\n-------------\n\n")

# Run the corrected function
await manager_run("What is the public sentiment and expert reviews about the Tesla Cybertruck?")

**User's Request:** What is the public sentiment and expert reviews about the Tesla Cybertruck?

---

### **short summary:**
Le Tesla Cybertruck suscite des avis très partagés en raison de son design futuriste et audacieux, salué pour son innovation mais critiqué pour ses difficultés de maniabilité et sa visibilité limitée. Les experts reconnaissent une courbe d'apprentissage notable à la conduite, tout en soulignant la place unique du véhicule sur le marché des pickups électriques.



------------



### **Full Report (Markdown):**
# Rapport d'analyse sur le Tesla Cybertruck

## Introduction
Le Tesla Cybertruck, avec son design disruptif et son positionnement innovant dans le segment des pickups électriques, suscite un intérêt considérable tant chez le public que chez les experts. Ce rapport synthétise les sentiments exprimés et les critiques techniques à son égard.

## Méthodologie
Les données ont été recueillies via l'analyse d'articles, de critiques d'experts automobiles et de retours clients, complétées par des rapports de tests réalisés par des sources reconnues comme Consumer Reports.

## Résultats

### Sentiment public
- Design futuriste et audacieux générant des opinions polarisées.
- Certains apprécient son image de robustesse et d'innovation.
- D'autres critiquent la praticité, notamment en raison de la visibilité arrière réduite et du maniement complexe.

### Avis d'experts
- Problèmes soulevés concernant la manoeuvrabilité, particulièrement en marche arrière.
- Ergonomie et visibilité diminuée lorsque la couverture de la benne est activée.
- Incident médiatisé de test de résistance du pare-brise, sans impact durable sur l'intérêt ou les précommandes.
- Une courbe d'apprentissage soulignée pour maîtriser la conduite et la direction unique.

## Discussion
Le Cybertruck incarne un compromis entre innovation stylistique et contraintes pratiques. Son esthétique « norm-busting » attire une niche préparée à accepter certains inconvénients pour un véhicule avant-gardiste. La visibilité et l'ergonomie restent des points à améliorer pour optimiser l'expérience utilisateur.

## Conclusion
Le Tesla Cybertruck se positionne comme un véhicule polarisant offrant une rupture technologique et stylistique majeure, tout en posant des défis d'utilisation au quotidien. Sa valeur repose autant sur sa capacité à redéfinir le segment que sur sa gestion des défauts opérationnels.

## Questions complémentaires
1. Quelles évolutions techniques sont prévues pour améliorer la visibilité et la maniabilité ?
2. Comment la clientèle cible perçoit-elle ces compromis entre style et fonctionnalité ?
3. Le Cybertruck pourra-t-il influencer durablement le marché des pickups électriques ?
4. Quel est l'impact des incidents médiatisés sur la confiance des consommateurs ?
5. Comment la conduite unique du Cybertruck se compare-t-elle à celle des concurrents ?



------------



### **Follow-up-Questions:**
-Quelles améliorations techniques Tesla envisage-t-elle pour résoudre les problèmes de visibilité et de maniabilité ?
-Comment le design polarisant du Cybertruck influence-t-il son adoption par les différents segments de clientèle ?
-Quel est l'impact des incidents publics sur la réputation à long terme du Cybertruck ?
-Comment la courbe d'apprentissage de la conduite affecte-t-elle l'expérience client ?
-Le Cybertruck change-t-il la dynamique du marché des pickups électriques face à ses concurrents ?



-------------



In [ ]:
# Define a pydantic model for the agent's output.
# This specifies exactly what fields the writer agent should return.

class FinalReport(BaseModel):
    short_summary: str  # A brief 3-3 sentence executive summary.
    markdown_report: str  # A detailed report in markdown format (at least 500 words)
    follow_up_questions: list[str]  # Alist of 3-5 suggested follow-up research questions.

# Create an AI agent called "Writer"
writer_agent = Agent(name = "Writer",
                     instructions = """
You are a senior market analyst tasked with generating an executive-level research report.

## Context
You will receive:
-The original user query (the research question or topic of interest)
-Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent'
 (for data analysis and key insights)

 You may call these helper tools as needed to gather additional information or clarify findings.

 ## Instructions
 Your job is to synthesize all available information and produce the following outputs:
 1.**Executive Summary:**Write a concise summary (2-3 sentences) highlighting the most important findings relevant to
  the original user query and the data gathered from the search snippets and analyst summaries.
  This summary should focus on capturing the essence of the overall research, giving readers a
  quick understanding of the report's key insights.
2.**Detailed Markdown Report:**Compose a comprehensive structured report in markdown format (at least 500 words)
  covering the analysis of the provided search snippets and summaries. Organize the content with clear headings
  and logical flows. Include sections such as:

Introduction: Briefly introduce the topic and the purpose of the report.
Methodology: Explain how the information was gathered and processed.
Findings: Present the key findings from the analysis, detailing patterns or significant insights from the data.
Discussion: Analyze what these findings mean in the context of the original query.
Conclusion: Summarize the overall insights and any recommendations based on the findings.

## Input
-The original user query.
-Summaries and analyses from the helper tools.

## Output
Return a structured object with:
-short_summary: The executive summary.
-markdown_report: The detailed markdown report.
-follow_up_questions: A list of 3-5 follow_up research questions.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport,

)

print("Writer AI Agent is now ready")

Writer AI Agent is now ready


In [ ]:
await manager_run("What is the public sentiment and expert reviews about the Tesla Cybertruck?")

**User's Request:** What is the public sentiment and expert reviews about the Tesla Cybertruck?

---

### **short summary:**
The Tesla Cybertruck has polarized public opinion with its bold, futuristic design, praised by some for innovation and toughness but criticized by others for its polarizing looks. Expert reviews highlight challenges with maneuverability and rear visibility, yet despite a notable glass strength test failure, its pre-orders and reputation remain strong. Consumer Reports notes a learning curve in handling but recognizes the Cybertruck as a standout in the electric pickup market, making it a unique vehicle blending avant-garde design with usability trade-offs.



------------



### **Full Report (Markdown):**
# Executive Report: Public Sentiment and Expert Reviews on Tesla Cybertruck

## Introduction
The Tesla Cybertruck represents a disruptive leap in electric vehicle design, entering the competitive pickup truck market with an unconventional, futuristic aesthetic. This report consolidates public sentiment and expert critiques to provide a comprehensive overview of perceptions surrounding this innovative vehicle.

## Methodology
This analysis synthesized information from multiple expert reviews, automotive journalism sources, consumer feedback, and performance evaluations, including notable assessments from Consumer Reports and industry experts such as Sandy Munro.

## Findings

### Public Sentiment
- The Cybertruck's exterior design is highly polarizing, often described as futuristic and norm-breaking.
- Enthusiasts admire its robust, innovative character and distinctive styling.
- Critics point out the vehicle's bold appearance can be off-putting, contributing to a divided consumer response.

### Expert Reviews
- Maneuverability challenges are recurrent themes, particularly difficulties with reversing and limited rear visibility exacerbated when the cargo bed cover is in use.
- The vehicle's unique steering and driving dynamics exhibit a learning curve, as highlighted in detailed road tests.
- A highly publicized failed glass strength demonstration initially raised concerns but had minimal long-term impact on public interest.

## Discussion
The Tesla Cybertruck exemplifies a trade-off between avant-garde design and functional practicality. Its unconventional styling sets it apart but also creates challenges for everyday usability. While some usability hurdles exist, Tesla's marketing and strong brand loyalty sustain high preorder volumes.

## Conclusion
The Cybertruck stands out as a pioneering electric pickup that polarizes opinions due to its design and handling characteristics. It remains a bold experiment that may shape future trends in electric trucks, balancing innovation with compromises in practicality.

## Recommendations
- Further refinements in rear visibility and maneuvering aids could enhance driver experience.
- Continued consumer education and support could mitigate the learning curve challenges.
- Monitoring consumer feedback post-launch will be critical to adapting features and improving usability.

## Follow-up Questions
1. What technical improvements are planned to address visibility and handling issues?
2. How does the Cybertruck's market performance compare to competing electric pickups?
3. What consumer demographics are most receptive to the Cybertruck’s design and features?
4. How have media incidents influenced Tesla's branding and consumer trust?
5. What innovations in electric truck design might be inspired by the Cybertruck’s market reception?



------------



### **Follow-up-Questions:**
-What technical improvements are planned to address visibility and handling issues in the Cybertruck?
-How does the Cybertruck compare to other electric pickups in terms of market performance and consumer satisfaction?
-What are the profiles of consumers most attracted to the Cybertruck’s unique design?
-What is the long-term impact of media incidents like the glass strength test on Tesla's brand?
-Which future electric truck innovations might be influenced by the Cybertruck’s polarizing design?



-------------



## **Practice Opportunity**
### 1. Run the pipeline again using the writer agent with French Language report generation capability (from the previous practice opportunity)
### 2. Create a new multi-agent team that writes tweets for each top campaigns:
### a. Define a creative Director Agent that brainstoms 2-3 ideas
### b. Define a strategist agent that selects the top 2 ideas and explains the reasoning.
### c. Define a copywriter agent that writes tweets for each top campaign.
### d. Set up any necessary tools for these agents.
### e. Build a pipeline where the output of each agent is passed to the next, similar to the previous example.
### Try running the full pipeline on a sample prompt (e.g "Launch campaign for a new eco-friendly water bottle in Bali")

## **Develop a Creative Advertising AI Agents Team**

# **Creative Director**

In [ ]:
session = SQLiteSession("creative_pipeline_practice")

class CreativeIdeas(BaseModel):
    ideas: list[str]

creative_director = Agent(name = "Creative_Director",
                          instructions = """

Context:
You are a creative at an advertising agency.

Instruction:
Brainstorm 3-5 creative ad campaign ideas for the given product.

Input:
The product name or description.

Output:
A list of 3-5 creative ideas (as bullet points or short phrases).
""",
    model = "gpt-4.1-mini",
    output_type = CreativeIdeas,
    )

# **Strategist**

In [ ]:
session = SQLiteSession("creative_pipeline_practice")

class SelectedCampaigns(BaseModel):
    top_campaigns: list[str]
    reasoning: str

strategist = Agent(name = "Creative_Director",
                          instructions = """

Context:
You are a marketing strategist evaluating campaign ideas.

Instruction:
From the provided list of campaign ideas, select the top 2 that are most promising and explain your reasoning.

Input:
A list of campaign ideas (one per line).

Output:
A list of the top 2 campaign ideas and a paragraph explaining why they stand out.
""",
    model = "gpt-4.1-mini",
    output_type = SelectedCampaigns,
)

# C**opywriter**

In [ ]:
session = SQLiteSession("creative_pipeline_practice")

class TweetCopy(BaseModel):
    tweets: list[str]

copy_writer = Agent(name = "CopyWriter",
                          instructions = """

Context:
You are a copywriter creating social media content.

Instruction:
For each of the top campaign ideas provided, write 2-3 engaging tweets that could be used to promote the campaign.

Input:
The selected top campaign ideas (one per line).

Output:
A list of 2-3 tweets for each campaign idea.
""",
    model = "gpt-4.1-mini",
    output_type = TweetCopy,
)

### **Let's build an Entire Pipeline i.e, A manager Function that is going to manage or orchestrate the entire workflow for us.**

In [ ]:
# Create a session to store and share information between agents.
session = SQLiteSession("research_agent_practice")

# Define an async function that takes the user's question and run the agents.
async def creative_pipeline(product_name: str):
    director_result = await Runner.run(creative_director, product_name, session = session)
    ideas = director_result.final_output.ideas

    strategist_input = "\n".join(ideas)
    strategist_result = await Runner.run(strategist, strategist_input, session = session)
    top_campaigns = strategist_result.final_output.top_campaigns

    tweets_input = "\n".join(top_campaigns)
    copywriter_result = await Runner.run(copy_writer, tweets_input, session = session)
    tweets = copywriter_result.final_output.tweets

    # Display Top Campaigns as separate bullet points
    print_markdown("### **Top Campaigns:**")
    for campaign in top_campaigns:
        print_markdown(f"- {campaign}")

    # Display Tweets as separate bullet points to clearly differentiate them
    print_markdown("### **Tweets:**")
    for tweet in tweets:
        print_markdown(f"- {tweet}")

### **Let's now run the Entire Pipeline or Manager Function**

In [ ]:
await creative_pipeline("Launch campaign for a new eco-friendly water bottle in Bali")

### **Top Campaigns:**

- Interactive beach clean-up events where participants get free bottles

- 'Bali's Breath' campaign highlighting the bottle's environmental impact with local ocean and forest imagery

### **Tweets:**

- Join us for a beach clean-up in Bali! 🌊 Grab your FREE eco-friendly water bottle and help protect our oceans and forests. Let’s make a difference together! #BalisBreath #EcoWarrior

- Every bottle counts! Swap single-use plastics for our 'Bali's Breath' eco-friendly bottle and be part of the change. Sip sustainably while embracing Bali’s natural beauty 🌿💧 #SustainableLiving #BalisBreath

- Feel the breath of Bali in every sip. Our new eco-friendly water bottle reduces plastic waste and celebrates Bali’s stunning ocean and forest landscapes. Drink clean, stay green! 🌊🌴 #BalisBreath #EcoFriendly

- Clean beaches, cleaner planet. Join our interactive beach clean-up, score a free eco bottle, and help keep Bali beautiful. Sign up now — because nature starts with us! #BeachCleanUp #BalisBreath

- From Bali’s forests to its shores, our eco-bottle carries the spirit of the island while protecting it too. Choose sustainability, choose 'Bali's Breath'. 🍃🌊 #ProtectBali #EcoWaterBottle